# Aula 05 — Formas Normais e Otimização Booleana

**Projeto:** Automação do Processo de Produção de Biodiesel (Transesterificação)  
**Objetivo:** estudar FND, FNC, Álgebra Booleana e a minimização da lógica de controle do processo.

> Este notebook foi elaborado diretamente a partir do material da Aula 05, mantendo as variáveis, expressões e sequência apresentadas no arquivo da aula.


## 1. Conceitos básicos

Vamos trabalhar com quatro variáveis booleanas:

- `A`: condição de processo 1
- `B`: condição de processo 2
- `C`: condição de processo 3
- `D`: condição de segurança

Em Álgebra Booleana:

- `1` = condição verdadeira/OK
- `0` = condição falsa/não OK
- `AND` (`∧`) = todas as condições precisam ser verdadeiras
- `OR` (`∨`) = pelo menos uma condição precisa ser verdadeira
- `NOT` (`¬`) = inversão da condição


## 2. Forma Normal Disjuntiva — FND / SOP

A FND representa uma lógica como uma **Soma de Produtos (SOP)**.  
Ela é formada por uma disjunção (`OR`) de mintermos.

No processo da aula:

- `A`: fluxo de óleo vegetal OK
- `B`: solução de metóxido pronta e dosada
- `C`: temperatura do reator entre 55 °C e 60 °C
- `D`: nível crítico do tanque de lavagem atingido

A expressão apresentada é:

\[
V_{reator,FND} =
(A B C \bar D)
+ (A B \bar C \bar D)
+ (A \bar B C \bar D)
\]

Em Python, podemos avaliar essa expressão para todas as combinações possíveis.


In [1]:
from itertools import product

def fnd(A, B, C, D):
    return (
        (A and B and C and not D)
        or (A and B and not C and not D)
        or (A and not B and C and not D)
    )

print("A B C D | V_reator")
print("------------------")

for A, B, C, D in product([0, 1], repeat=4):
    V = int(fnd(A, B, C, D))
    print(A, B, C, D, "|", V)


A B C D | V_reator
------------------
0 0 0 0 | 0
0 0 0 1 | 0
0 0 1 0 | 0
0 0 1 1 | 0
0 1 0 0 | 0
0 1 0 1 | 0
0 1 1 0 | 0
0 1 1 1 | 0
1 0 0 0 | 0
1 0 0 1 | 0
1 0 1 0 | 1
1 0 1 1 | 0
1 1 0 0 | 1
1 1 0 1 | 0
1 1 1 0 | 1
1 1 1 1 | 0


### Interpretação

A saída `V_reator = 1` somente ocorre nas combinações previstas pelos três mintermos da FND.

O termo `¬D` aparece em todos eles. Portanto, a condição de segurança relacionada ao nível crítico é comum aos casos de habilitação apresentados.


## 3. Forma Normal Conjuntiva — FNC / POS

A FNC representa a lógica como um **Produto de Somas (POS)**.

Na aula, ela é usada para representar requisitos de travamento de segurança (*interlocks*).

A expressão fornecida é:

\[
V_{reator,FNC} =
(A+B+C+\bar D)
(A+B+\bar C+\bar D)
(A+\bar B+C+\bar D)
(\bar A+B+C+\bar D)
\]

Uma forma prática de estudar essa expressão é montar uma tabela-verdade e verificar o resultado para todas as combinações.


In [2]:
def fnc(A, B, C, D):
    return (
        (A or B or C or not D)
        and (A or B or not C or not D)
        and (A or not B or C or not D)
        and (not A or B or C or not D)
    )

print("A B C D | V_reator_FNC")
print("---------------------")

for A, B, C, D in product([0, 1], repeat=4):
    V = int(fnc(A, B, C, D))
    print(A, B, C, D, "|", V)


A B C D | V_reator_FNC
---------------------
0 0 0 0 | 1
0 0 0 1 | 0
0 0 1 0 | 1
0 0 1 1 | 0
0 1 0 0 | 1
0 1 0 1 | 0
0 1 1 0 | 1
0 1 1 1 | 1
1 0 0 0 | 1
1 0 0 1 | 0
1 0 1 0 | 1
1 0 1 1 | 1
1 1 0 0 | 1
1 1 0 1 | 1
1 1 1 0 | 1
1 1 1 1 | 1


## 4. Leis da Álgebra Booleana

### Idempotência
\[
p \land p = p
\]
\[
p \lor p = p
\]

Remove repetições desnecessárias.

### Absorção
\[
p \lor (p \land q)=p
\]

Um termo mais restritivo pode ser absorvido por um termo mais geral.

### Elemento inverso
\[
p\land\neg p=0
\]
\[
p\lor\neg p=1
\]

Uma variável ou sua negação representam condições complementares.

### De Morgan
\[
\neg(p\land q)=\neg p\lor\neg q
\]

### Consenso
\[
(pq)+(\neg pr)+(qr)=(pq)+(\neg pr)
\]

A aula apresenta o consenso como uma ferramenta para eliminar termos redundantes e reduzir a lógica.


## 5. Aplicação: válvula de dosagem de reagente

Agora usamos a segunda situação apresentada na aula.

Variáveis:

- `A`: pressão no reator OK
- `B`: temperatura do reator OK (55 °C–60 °C)
- `C`: válvula de entrada de óleo aberta
- `D`: parada de emergência ativada (`D = 1` significa emergência)

A lógica bruta é:

\[
V_{reagente} =
(\bar DABC)
+(\bar DAB\bar C)
+(\bar DA\bar BC)
\]


In [3]:
def v_reagente_bruta(A, B, C, D):
    return (
        (not D and A and B and C)
        or (not D and A and B and not C)
        or (not D and A and not B and C)
    )

print("A B C D | V_bruta")
print("----------------")

for A, B, C, D in product([0, 1], repeat=4):
    print(A, B, C, D, "|", int(v_reagente_bruta(A, B, C, D)))


A B C D | V_bruta
----------------
0 0 0 0 | 0
0 0 0 1 | 0
0 0 1 0 | 0
0 0 1 1 | 0
0 1 0 0 | 0
0 1 0 1 | 0
0 1 1 0 | 0
0 1 1 1 | 0
1 0 0 0 | 0
1 0 0 1 | 0
1 0 1 0 | 1
1 0 1 1 | 0
1 1 0 0 | 1
1 1 0 1 | 0
1 1 1 0 | 1
1 1 1 1 | 0


## 6. Minimização passo a passo

### Passo 1 — agrupar os dois primeiros termos

\[
\bar DABC+\bar DAB\bar C
\]

Colocando os fatores comuns em evidência:

\[
(\bar DAB)(C+\bar C)
\]

### Passo 2 — elemento inverso

Como:

\[
C+\bar C=1
\]

temos:

\[
(\bar DAB)\cdot1=\bar DAB
\]

Portanto:

\[
V_{reagente}=\bar DAB+\bar DA\bar BC
\]

### Passo 3 — fatoração de \(\bar DA\)

\[
V_{reagente}=(\bar DA)[B+\bar BC]
\]

### Passo 4 — simplificação

Usando:

\[
B+\bar BC=B+C
\]

obtemos a forma otimizada:

\[
\boxed{V_{otim}=\bar D A(B+C)}
\]


In [4]:
def v_reagente_otimizada(A, B, C, D):
    return (not D) and A and (B or C)

print("Comparação entre lógica bruta e otimizada")
print("A B C D | Bruta | Otimizada | Iguais?")
print("-----------------------------------------")

todas_iguais = True

for A, B, C, D in product([0, 1], repeat=4):
    bruta = int(v_reagente_bruta(A, B, C, D))
    otim = int(v_reagente_otimizada(A, B, C, D))
    iguais = bruta == otim
    todas_iguais &= iguais
    print(A, B, C, D, "|", bruta, "    |", otim, "       |", iguais)

print("\nAs duas expressões são equivalentes?", todas_iguais)


Comparação entre lógica bruta e otimizada
A B C D | Bruta | Otimizada | Iguais?
-----------------------------------------
0 0 0 0 | 0     | 0        | True
0 0 0 1 | 0     | 0        | True
0 0 1 0 | 0     | 0        | True
0 0 1 1 | 0     | 0        | True
0 1 0 0 | 0     | 0        | True
0 1 0 1 | 0     | 0        | True
0 1 1 0 | 0     | 0        | True
0 1 1 1 | 0     | 0        | True
1 0 0 0 | 0     | 0        | True
1 0 0 1 | 0     | 0        | True
1 0 1 0 | 1     | 1        | True
1 0 1 1 | 0     | 0        | True
1 1 0 0 | 1     | 1        | True
1 1 0 1 | 0     | 0        | True
1 1 1 0 | 1     | 1        | True
1 1 1 1 | 0     | 0        | True

As duas expressões são equivalentes? True


## 7. Interpretação física da expressão otimizada

\[
V_{otim}=\bar D A(B+C)
\]

A expressão pode ser lida como:

1. `¬D`: **não existe parada de emergência**;
2. `A`: **a pressão do reator está OK**;
3. `(B + C)`: **a temperatura está OK OU a válvula de entrada de óleo está aberta**.

Assim, a estrutura principal de segurança é:

\[
\boxed{\text{Sem emergência} \land \text{Pressão OK}}
\]

e depois é avaliada a condição:

\[
\boxed{\text{Temperatura OK} \lor \text{Entrada de óleo aberta}}
\]


## 8. Comparação da complexidade

Segundo a aula, a expressão original possui **11 operadores lógicos**, enquanto a forma otimizada possui apenas **3 operadores lógicos**.

A forma otimizada é:

\[
\boxed{\bar D \land A \land (B\lor C)}
\]

Isso representa uma redução importante para implementação em CLP, pois a aula destaca:

- menor complexidade;
- menor tempo de varredura (*scan time*);
- menor ocupação de memória;
- implementação mais simples em Ladder;
- implementação mais simples em FBD;
- manutenção do *interlock* mestre de segurança.


## 9. Exercício de fixação

Altere os valores de `A`, `B`, `C` e `D` no código abaixo e observe quando a válvula é acionada.

Perguntas:

1. O que acontece quando `D = 1`?
2. O que acontece quando `A = 0`?
3. Se `A = 1` e `D = 0`, quais combinações de `B` e `C` permitem o acionamento?


In [5]:
# Teste livre
A = 1
B = 0
C = 1
D = 0

saida = int(v_reagente_otimizada(A, B, C, D))

print("A =", A)
print("B =", B)
print("C =", C)
print("D =", D)
print("V_reagente =", saida)


A = 1
B = 0
C = 1
D = 0
V_reagente = 1
